<a href="https://colab.research.google.com/github/mybenkhadda/GEMS---drivendata/blob/main/notebooks/colab/GEMS_DISCOVERY_GEOMOE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GEMS discovery-first GeoMoE (code rev dgm.1-r1)

**One experiment, one CONFIG, run top to bottom.** Candidate discovery -> geological experts -> static / rule / learned gating -> official DTI under three validation scenarios -> a validated GeoTIFF **only if** the predeclared selection rule selects a model.

| | |
|---|---|
| Runtime | Colab **CPU**, High-RAM recommended. No GPU, no neural network is trained. |
| Inputs (read-only) | the frozen B0 / B1 / W2 artifacts the submission factory attaches (`outputs/p6_fast_track/...`, experiments, folds, template), G01/G02 run, cached M7 DEM mosaic if present |
| Outputs | `MyDrive/GEMS_2/outputs/discovery_geomoe/` (audit, validation, candidates, experts, gating, predictions, submissions, figures, reports, manifests) |
| Code | `gems_bench/geomoe/` (`run.DiscoveryGeoMoE`), config `configs/geomoe/discovery_geomoe.yaml`, docs `docs/geomoe/DISCOVERY_GEOMOE.md` |

Scenarios: **V1** whole withheld systems (existing discovery splits), **V2** network completion (real mapped tips / branches / splays / sub-sections withheld inside each geo fold), **V3** geographic folds. Models: **A_pixel** frozen W2 (reference, public 0.0942), **A_seg** W2 on the unified pool, **B** static fusion, **C** rule gate (+ ablations), **D** learned gate (only if the coverage gate allows it).

**After a disconnect:** restart the session and run every cell again. Finished stages print `SKIP_CACHED`; finished candidate units print `reused`.

## 1 - Configuration (the only cell you edit)

`mode`: `FULL` | `DIAGNOSTIC` (everything except the GeoTIFF) | `RESUME` (= FULL, cache forced) | `REPORT_ONLY` (attach + audit + reports). `experts`: switch a family's layers off (`dem`: `cached_only` never downloads). `experiments.learned_gate`: `conditional` (stage 06 decides), `True` (forced, recorded), `False`. `export.exploratory`: also write a clearly labelled EXPLORATORY file when the rule retains W2.

In [ ]:
CONFIG = {
    'repo': 'mybenkhadda/GEMS---drivendata',
    'ref': 'main',                          # pin the pushed commit SHA for a reproducible run
    'drive_root': '/content/drive/MyDrive/GEMS_2',
    'local_root': '/content/gems_work',

    'mode': 'FULL',                         # FULL | DIAGNOSTIC | RESUME | REPORT_ONLY
    'reuse_cached': True,
    'device': 'cpu',

    'experts': {
        'cnn': True,
        'w2': True,
        'magnetic_gravity': True,
        'g01_g02': True,
        'dem': 'cached_only',               # cached_only | off  (never a DEM download)
        'network': True,
    },
    'experiments': {
        'validation_audit': True,
        'candidate_ceiling': True,
        'static_fusion': True,
        'rule_gate': True,
        'learned_gate': 'conditional',      # conditional | True | False
        'deployment_comparison': True,
    },
    'export': {
        'enabled': True,
        'only_validated': True,
        'exploratory': False,               # True: also write an EXPLORATORY file when W2 is retained
        'manual_upload': True,              # must stay True
    },
    'save_png': True,
    'sync_to_drive': True,
    'overrides': {},                        # optional overrides of configs/geomoe/discovery_geomoe.yaml
}
PRIVATE_REPO = False
RUN_PYTEST = True                           # run tests/test_discovery_geomoe.py (synthetic) before any export
EXPECT = {'code_rev': 'dgm.1-r1', 'commit': None}   # set 'commit' to the pushed SHA

## 2 - Environment, Drive, frozen engine

Mounts Drive, clones the pinned ref, installs the package, builds the submission factory (the existing frozen-engine loader; it exports nothing here) and the GeoMoE runner. The runtime must be fresh (`gems_bench` not yet imported).

In [ ]:
import base64, os, subprocess, sys
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
CODE = '/content/GEMS'
REPO_URL = 'https://github.com/' + CONFIG['repo'] + '.git'

def _git_env():
    if not PRIVATE_REPO:
        return None
    from google.colab import userdata
    hdr = base64.b64encode(('x-access-token:' + (userdata.get('GITHUB_TOKEN') or '')).encode()).decode()
    return {**os.environ, 'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
            'GIT_CONFIG_VALUE_0': 'AUTHORIZATION: basic ' + hdr}

def git(*args, auth=False, check=True):
    r = subprocess.run(['git', *args], capture_output=True, text=True, env=_git_env() if auth else None)
    if check and r.returncode:
        raise RuntimeError('git ' + ' '.join(args[:4]) + ' failed:\n' + r.stderr[-1500:])
    return r

if not Path(CODE, '.git').exists():
    git('clone', '-q', REPO_URL, CODE, auth=True)
git('-C', CODE, 'fetch', '-q', '--tags', 'origin', auth=True)
ref = CONFIG['ref']
if git('-C', CODE, 'rev-parse', '--verify', '-q', 'refs/remotes/origin/' + ref, check=False).returncode == 0:
    git('-C', CODE, 'checkout', '-q', '-B', 'geomoe_colab_active', 'origin/' + ref)
else:
    git('-C', CODE, 'checkout', '-q', '--detach', ref)
HEAD = git('-C', CODE, 'rev-parse', 'HEAD').stdout.strip()
if 'gems_bench' in sys.modules:
    raise RuntimeError('gems_bench already imported: Runtime -> Restart session, then run from the top')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', CODE + '[colab]'], check=True)
sys.path.insert(0, CODE)
os.environ['GEMS_CODE_COMMIT'] = HEAD

import pandas as pd
from IPython.display import Image, Markdown, display
import gems_bench.geomoe as GM
from gems_bench.geomoe import run as GR, selftest as GST
from gems_bench.leaderboard_focus import factory as FA
assert GM.GEOMOE_CODE_REV == EXPECT['code_rev'], f'code rev {GM.GEOMOE_CODE_REV} != {EXPECT["code_rev"]}'
assert not EXPECT['commit'] or HEAD.startswith(EXPECT['commit']), f'HEAD {HEAD} != {EXPECT["commit"]}'
NB = GR.notebook_config(CONFIG)
F = FA.Factory(GR.factory_config(NB), HEAD)     # frozen-engine loader only (no factory candidate is exported)
G = GR.DiscoveryGeoMoE(CONFIG, HEAD, factory=F)
RUN_ALL = G.run                                   # optional one-call entry point == cells 4-16 below
OUT = G.ws.root

def show(rel, cols=None, n=40):
    p = OUT / rel
    if not p.exists():
        print('ABSENT:', rel); return None
    df = pd.read_csv(p) if p.suffix == '.csv' else pd.read_parquet(p)
    display(df[[c for c in (cols or df.columns) if c in df]].head(n)); return df

print('SHA', HEAD, '| code rev', GM.GEOMOE_CODE_REV, '| mode', NB['mode'])
print('Drive outputs:', G.drive / G.out)

## 3 - Synthetic self-tests (before anything else)

GeoMoE contracts (V2 withholding uses real pixels only, leakage detection, FastScorer == official reference, known-fault masking, distance bands, dedup, provenance, oracle accounting, missing experts never zero, cross-fitted normalisation, gate weights + analytic gradient, collapse, PU targets, fair budgets, rule gate, falsifiable selection, coverage gate, read-back contract) + the pytest file. Stage 12 re-runs them and refuses to export on a critical failure.

In [ ]:
ST = GST.run_selftests()
G.ws.save_table(ST, 'audit/selftests.csv')
display(ST[['test', 'critical', 'status', 'detail']])
print(f"{int((ST.status == 'PASS').sum())}/{len(ST)} PASS")
if RUN_PYTEST:
    r = subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_discovery_geomoe.py'], cwd=CODE,
                       capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-1500:])
    G.ws.save_text(r.stdout + '\n' + r.stderr, 'audit/pytest_output.txt')

## 4 - [01] Attach existing artifacts

Factory self-tests, read-only preflight, frozen engine on the CPU. A BLOCKED attach stops the modeling stages; the reports still explain why.

In [ ]:
M01 = G.s01_attach()
print(M01['status'], M01.get('note'), M01.get('reasons', ''))

## 5 - [02] Audit evaluation and leakage (reuse matrix)

`component | available | executed | reusable | limitation | action`, source compatibility, and the recorded historical numbers with their protocol (context only; never mixed with this run's numbers).

In [ ]:
G.s02_audit()
show('audit/component_inventory.csv'); show('audit/protocol_comparison.csv')

## 6 - [03] Build discovery validation scenarios

V1 / V2 / V3 units, V2 withheld events (real mapped pixels), leakage checks. Any failed leakage check marks the stage FAILED and blocks selection.

In [ ]:
G.s03_scenarios()
show('validation/scenarios.csv')
L = show('audit/label_leakage_audit.csv', n=5)
if L is not None: print('leakage failures:', int((~L.ok.astype(bool)).sum()))

## 7 - [04] Build unified candidates

Per unit (one at a time, memory released): label-free source layers -> low-threshold 1-px pools -> deduplicated, split, provenance-tagged inventory -> segment table + coverage parts + oracle curve. The longest CPU stage; finished units are reused after a disconnect.

In [ ]:
G.s04_candidates()
show('candidates/source_provenance.csv', n=60)

## 8 - [05] Compute oracle coverage (+ W2 baseline, metric equivalence)

W2 (A_pixel) under the shared grid, compared cell by cell with the frozen boost-A grid (must match). Coverage of every pool, unique / missed systems, coverage by subtype, M7 vs X2 under one definition. Oracle rows are diagnostics and never reach a selection.

In [ ]:
G.s05_oracle()
E = show('audit/metric_equivalence.csv', n=5)
if E is not None and 'status' in E: print('metric equivalence:', E.status.value_counts().to_dict())
show('candidates/oracle_coverage.csv'); show('audit/protocol_comparison_measured.csv')

## 9 - [06] Decide: generation vs ranking bottleneck

Predeclared gate (config `candidates.gate`). GENERATION -> the learned gate is not trained (it cannot recover systems no candidate covers).

In [ ]:
G.s06_decision()
import json; print(json.dumps(G.ws.read_json('candidates/coverage_decision.json'), indent=1, default=str)[:3000])

## 10 - [07] Build available expert evidence

Expert scores per candidate, availability (an absent source is UNAVAILABLE, never zero), cross-fitted ECDF normalisation per (scenario, held-out zone).

In [ ]:
G.s07_experts()
A = show('experts/source_availability.csv', n=0)
if A is not None: display(A[A.member == '*'])

## 11 - [08] Evaluate fixed fusion (Model B) and W2 on the same pool (A_seg)

In [ ]:
G.s08_static()
show('gating/static_fusion_results.csv', ['scenario', 'model', 'dti', 'worst_zone_dti', 'params_all'])

## 12 - [09] Evaluate rule-based geological gate (Model C + one ablation per rule)

In [ ]:
G.s09_rules()
show('gating/rule_gate_results.csv', ['scenario', 'model', 'dti', 'worst_zone_dti']); show('gating/rule_firing_rates.csv')

## 13 - [10] Train learned gate if justified, then the paired comparison and model selection

The comparison and the predeclared selection always run (with whatever models exist).

In [ ]:
G.s10_learned()
show('gating/model_summary.csv', ['scenario', 'model', 'dti', 'worst_zone_dti', 'worst_zone', 'fold_std'])
show('gating/paired_comparison.csv')
print(json.dumps(G.ws.read_json('gating/selection.json'), indent=1, default=str)[:2500])

## 14 - [11] Compare deployment strategies (V3, cross-fitted, same cells)

In [ ]:
G.s11_deployment()
show('gating/deployment_comparison.csv'); show('gating/deployment_parity.csv')

## 15 - [12] Produce a validated GeoTIFF if justified

Only the selected model (or an EXPLORATORY one when asked). Factory / P6 writer, official checks, recipe plan audit, read-back of the file on disk, sha256. Nothing is uploaded.

In [ ]:
G.s12_export()
show('submissions/submission_manifest.csv')
p = OUT / 'submissions/README_UPLOAD.md'
if p.exists(): display(Markdown(p.read_text()))

## 16 - [13] Save reports and figures

Written from persisted tables only (re-runnable alone after a restart: run cells 1-2, then `G.s13_reports()`).

In [ ]:
G.s13_reports()
for f in ('candidate_coverage', 'unique_recovery', 'discovery_vs_geography', 'per_zone_dti', 'prediction_case_studies'):
    p = OUT / 'figures' / f'{f}.png'
    if p.exists(): display(Image(str(p)))
display(Markdown((OUT / 'reports/SCIENTIFIC_DECISION.md').read_text()))
display(Markdown((OUT / 'reports/NEXT_ACTION.md').read_text()))
print('stages:', G.status)